# Optimisation des approvisionnements — Modèle simplifié à un seul niveau

## Contexte

Ce modèle traite un problème d'approvisionnement pour un ensemble de références (`PN`) sur un
horizon de plusieurs mois. Pour chaque référence, il faut décider **quand commander** et
**quelle quantité**, de façon à satisfaire la demande à chaque période, tout en respectant :

- la **quantité minimale de commande** (MOQ) imposée par le fournisseur,
- le **délai de livraison** (lead time) entre la commande et la réception,
- un **stock de sécurité** minimal à partir du 2ᵉ mois (le stock initial est une donnée subie,
  pas une décision — voir section 3),

et en minimisant le **coût total** : coût de possession du stock (capital immobilisé) + coût
fixe de passation de chaque commande.

Ce modèle est volontairement à **un seul niveau** (pas de nomenclature, pas de distinction
matière première / semi-fini / produit fini) : chaque référence est traitée indépendamment,
ce qui permet une optimisation rapide et une lecture directe des résultats.

## Démarche (identique à la structure AMPL/GUSEK)

1. **Paramètres** — déclarés sans valeur
2. **Variables**
3. **Fonction objectif** (+ contraintes)
4. **Solve**
5. **Mettre en place la data** — ici, directement dans le notebook (pas de fichier Excel)


In [ ]:
from pyomo.environ import *
from IPython.display import display
import pandas as pd
import numpy as np
import math


---
## 1. LES PARAMÈTRES


In [ ]:
model = AbstractModel()

# ---------- Horizon ----------
model.T   = Param(within=PositiveIntegers)
model.PER = RangeSet(1, model.T)

# ---------- Références ----------
model.I = Set()

# ---------- Paramètres généraux ----------
model.M = Param()   # big-M, pour la liaison MOQ / variable binaire

# ---------- Paramètres par référence ----------
model.prix          = Param(model.I)
model.stock_init    = Param(model.I)
model.LT            = Param(model.I)   # lead time (en mois)
model.MOQ           = Param(model.I)
model.SS            = Param(model.I)   # stock de sécurité
model.cout_stockage = Param(model.I)   # coût de possession par unité et par mois
model.cout_commande = Param(model.I)   # coût fixe de passation d'une commande

model.demande = Param(model.I, model.PER)


---
## 2. LES VARIABLES


In [ ]:
model.S = Var(model.I, model.PER, domain=NonNegativeReals)   # stock en fin de période
model.O = Var(model.I, model.PER, domain=NonNegativeReals)   # quantité commandée
model.z = Var(model.I, model.PER, domain=Binary)              # commande déclenchée (pour MOQ)


---
## 3. LA FONCTION OBJECTIF (et les contraintes)

**C1 — Bilan de stock.** Le stock d'une période est le stock précédent, plus la réception des
commandes passées `LT` périodes plus tôt, moins la demande. Le stock initial (t=1) est une
donnée subie, pas une décision.

**C2 — MOQ.** Si une commande est passée (`z=1`), elle doit être au moins égale au MOQ ; si
`z=0`, aucune commande n'est possible (liaison par grand-M).

**C3 — Stock de sécurité**, à partir de t=2 (même raisonnement que pour le stock initial : on
ne peut pas exiger que le stock de départ respecte déjà le seuil de sécurité).


In [ ]:
def cost_rule(m):
    cout_possession = sum(m.cout_stockage[i]*m.S[i,t] for i in m.I for t in m.PER)
    cout_lancement   = sum(m.cout_commande[i]*m.z[i,t]  for i in m.I for t in m.PER)
    return cout_possession + cout_lancement
model.cost = Objective(rule=cost_rule, sense=minimize)


In [ ]:
# C1 — Bilan de stock
def c1_rule(m, i, t):
    S_prev = m.stock_init[i] if t == 1 else m.S[i, t-1]
    t_cmd  = t - m.LT[i]
    reception = m.O[i, t_cmd] if t_cmd >= 1 else 0
    return m.S[i,t] == S_prev + reception - m.demande[i,t]
model.C1_Bilan = Constraint(model.I, model.PER, rule=c1_rule)

# C2 — MOQ (quantité minimale de commande) et liaison binaire
def c2a_rule(m, i, t):
    return m.O[i,t] >= m.MOQ[i] * m.z[i,t]
model.C2a_MOQ_min = Constraint(model.I, model.PER, rule=c2a_rule)

def c2b_rule(m, i, t):
    return m.O[i,t] <= m.M * m.z[i,t]
model.C2b_MOQ_max = Constraint(model.I, model.PER, rule=c2b_rule)

# C3 — Stock de sécurité, à partir de t=2
def c3_rule(m, i, t):
    if t == 1:
        return Constraint.Skip
    return m.S[i,t] >= m.SS[i]
model.C3_StockSecurite = Constraint(model.I, model.PER, rule=c3_rule)


---
## 4. SOLVE


In [ ]:
def resoudre(instance, solveur="cbc", verbose=True):
    opt = SolverFactory(solveur)
    return opt.solve(instance, tee=verbose)


In [ ]:
import shutil
chemin_cbc = shutil.which("cbc")
if chemin_cbc is None:
    import subprocess
    subprocess.run(["apt-get", "install", "-y", "coinor-cbc"], check=True)
    chemin_cbc = shutil.which("cbc")
assert chemin_cbc is not None, "CBC introuvable"
print(f"CBC disponible : {chemin_cbc}")


---
## 5. METTRE EN PLACE LA DATA

Les données sont saisies directement dans le notebook (pas de fichier Excel), sur un horizon
de 12 mois et 6 références. La demande est volontairement variable dans le temps (saisonnalité
légère) pour illustrer un cas réaliste.

### 5.1 Références et demande mensuelle


In [ ]:
noms_mois = {1:"Jan",2:"Fév",3:"Mar",4:"Avr",5:"Mai",6:"Juin",
             7:"Juil",8:"Août",9:"Sep",10:"Oct",11:"Nov",12:"Déc"}
T_max = 12

# Prix unitaire, lead time (mois), MOQ, coût fixe de commande — par référence
references = {
    "PN-A": {"prix": 120, "LT": 1, "MOQ": 50,  "cout_commande": 200},
    "PN-B": {"prix": 45,  "LT": 2, "MOQ": 200, "cout_commande": 150},
    "PN-C": {"prix": 300, "LT": 1, "MOQ": 20,  "cout_commande": 400},
    "PN-D": {"prix": 15,  "LT": 1, "MOQ": 500, "cout_commande": 100},
    "PN-E": {"prix": 80,  "LT": 3, "MOQ": 100, "cout_commande": 250},
    "PN-F": {"prix": 200, "LT": 2, "MOQ": 30,  "cout_commande": 300},
}
refs = list(references.keys())

# Demande mensuelle (unités) — légère saisonnalité (creux en été, pic en fin d'année)
profil_saison = {1:1.0,2:0.95,3:1.0,4:1.0,5:0.9,6:0.8,7:0.7,8:0.7,9:0.95,10:1.05,11:1.15,12:1.2}
demande_base = {"PN-A": 50, "PN-B": 200, "PN-C": 20, "PN-D": 500, "PN-E": 100, "PN-F": 30}

demande = {
    (i, t): round(demande_base[i] * profil_saison[t])
    for i in refs for t in range(1, T_max + 1)
}

df_demande = pd.DataFrame({i: [demande[(i,t)] for t in range(1,T_max+1)] for i in refs},
                           index=[noms_mois[t] for t in range(1,T_max+1)]).T
print("Demande mensuelle par référence :")
display(df_demande)


### 5.2 Stock initial et stock de sécurité

**Garantie de faisabilité.** Entre t=1 et t=LT[i], aucune réception n'est possible (le modèle
ne peut décider d'une commande qu'à partir de t=1, qui arrive au plus tôt en t=1+LT[i]) : le
stock ne peut donc que décroître pendant cette fenêtre, son point le plus bas étant exactement
en t=LT[i]. En imposant :

`stock_init[i] >= SS[i] + somme(demande[i,t] pour t=1..LT[i])`

le stock reste garanti au-dessus du seuil de sécurité pendant toute la fenêtre sans
réception — et au-delà, le modèle a toute latitude pour commander (aucune limite haute sur le
stock dans cette version simplifiée), donc la faisabilité est assurée **par construction**,
quelle que soit la solution optimale trouvée par le solveur.

Le stock de sécurité est calculé par la formule classique `SS = z_alpha * sigma_D * sqrt(LT)`.


In [ ]:
tau, z_alpha = 0.20, 1.65   # taux de possession annuel, niveau de service ~95%

demande_moyenne = {i: np.mean([demande[(i,t)] for t in range(1, T_max+1)]) for i in refs}
sigma_D         = {i: np.std([demande[(i,t)] for t in range(1, T_max+1)]) for i in refs}

SS = {i: round(z_alpha * sigma_D[i] * math.sqrt(references[i]["LT"])) for i in refs}

# Stock initial : garantit SS atteint pendant toute la fenêtre sans réception (t=1..LT[i])
MARGE = 1.05
stock_init = {}
for i in refs:
    LT_i = references[i]["LT"]
    demande_fenetre = sum(demande[(i, t)] for t in range(1, LT_i + 1))
    stock_init[i] = round((SS[i] + demande_fenetre) * MARGE)

df_init = pd.DataFrame({"Stock initial": stock_init, "Stock de sécurité (SS)": SS,
                         "Lead time (mois)": {i: references[i]["LT"] for i in refs}})
display(df_init)


### 5.3 Construction du `DataPortal` et de l'instance concrète

In [ ]:
cout_stockage = {i: round(tau * references[i]["prix"] / 12, 4) for i in refs}  # coût mensuel

M_bigM = max(demande_base.values()) * T_max * 3   # grand-M largement suffisant

data = DataPortal()
data['T'] = {None: T_max}
data['I'] = refs
data['M'] = {None: M_bigM}

data['prix']          = {i: references[i]["prix"] for i in refs}
data['stock_init']    = stock_init
data['LT']             = {i: references[i]["LT"]  for i in refs}
data['MOQ']            = {i: references[i]["MOQ"] for i in refs}
data['SS']             = SS
data['cout_stockage']  = cout_stockage
data['cout_commande']  = {i: references[i]["cout_commande"] for i in refs}
data['demande']        = demande

instance = model.create_instance(data)
print("Instance créée :", len(refs), "références,", T_max, "mois")


### 5.4 Résolution

In [ ]:
resultats = resoudre(instance, solveur="cbc", verbose=True)
print("\nStatut :", resultats.solver.termination_condition)
print("Coût total optimal :", round(value(instance.cost), 2), "$")


### 5.5 Résultats — plan d'approvisionnement

In [ ]:
lignes = []
for i in refs:
    for t in range(1, T_max+1):
        lignes.append({
            "PN": i, "Mois": noms_mois[t],
            "Demande": demande[(i,t)],
            "Commande (O)": round(value(instance.O[i,t])),
            "Stock fin de mois (S)": round(value(instance.S[i,t])),
            "Commande déclenchée (z)": int(round(value(instance.z[i,t]))),
        })
df_resultats = pd.DataFrame(lignes)

for i in refs:
    print(f"\n--- {i} ---")
    display(df_resultats[df_resultats["PN"] == i].drop(columns="PN").set_index("Mois"))


### 5.6 Graphique — évolution du stock par référence

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for i in refs:
    valeurs = [value(instance.S[i,t]) for t in range(1, T_max+1)]
    ax.plot([noms_mois[t] for t in range(1, T_max+1)], valeurs, marker="o", label=i)

ax.set_title("Évolution du stock par référence")
ax.set_xlabel("Mois"); ax.set_ylabel("Stock (unités)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 5.7 Résumé

In [ ]:
cout_possession_total = sum(cout_stockage[i]*value(instance.S[i,t]) for i in refs for t in range(1,T_max+1))
cout_lancement_total  = sum(references[i]["cout_commande"]*value(instance.z[i,t]) for i in refs for t in range(1,T_max+1))
nb_commandes           = sum(int(round(value(instance.z[i,t]))) for i in refs for t in range(1,T_max+1))

print("=" * 50)
print("  RÉSUMÉ")
print("=" * 50)
print(f"  Coût total optimal      : {value(instance.cost):,.2f} $")
print(f"    dont possession stock : {cout_possession_total:,.2f} $")
print(f"    dont lancement cmd    : {cout_lancement_total:,.2f} $")
print(f"  Nombre de commandes     : {nb_commandes}")
print("=" * 50)
